# Chest X-ray — SMALL run (C3 go/no-go) — CRASH-SAFE

**Why 3 models, not 7?** Cheap go/no-go: answers ONE question — does SRC predict cross-source collapse? GO -> run the 7-model notebook. NO-GO -> you saved hours. (Want 7 from the start? use `colab_full.ipynb`.)

**Crash-safe:** Cell 3 symlinks results/ to Drive; training backs up every epoch to Drive. Disconnect = minutes lost, not hours. Reconnect, re-run, it resumes.

## Cell 1 — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone / update code

In [ ]:
import os
REPO='https://github.com/kamlishgoswami/Chest_Xray.git'
if not os.path.exists('/content/Chest_Xray'):
    !git clone $REPO /content/Chest_Xray
else:
    !git -C /content/Chest_Xray pull
os.chdir('/content/Chest_Xray'); print('cwd:', os.getcwd())

## Cell 3 — CRASH-SAFETY: point results/ AT Google Drive (BEFORE anything writes)
Every checkpoint, JSON and figure now lands on Drive the INSTANT it is written. A disconnect no longer loses results — reconnect, re-run, and `--resume` skips finished models.

In [ ]:
import os, shutil
DRIVE='/content/drive/MyDrive/cxr_data'
RES=f'{DRIVE}/results'; BACKUP=f'{DRIVE}/backup'
os.makedirs(RES,exist_ok=True); os.makedirs(BACKUP,exist_ok=True)
if os.path.islink('results'): os.remove('results')
elif os.path.isdir('results'): shutil.rmtree('results')
os.symlink(RES,'results')
print('results ->', os.path.realpath('results')); print('backup  ->', BACKUP)

## Cell 4 — unzip data to fast local disk (slow, once per session)

In [ ]:
import os, glob
DRIVE='/content/drive/MyDrive/cxr_data'
os.makedirs('data/raw',exist_ok=True)
for z in glob.glob(f'{DRIVE}/*.zip'):
    print('unzipping', os.path.basename(z), flush=True)
    !unzip -q -o "$z" -d data/raw
print('folders:', os.listdir('data/raw'))

## Cell 5 — bring the manifest

In [ ]:
import os, shutil
DRIVE='/content/drive/MyDrive/cxr_data'
if os.path.exists(f'{DRIVE}/manifest.csv'):
    shutil.copy(f'{DRIVE}/manifest.csv','data/manifest.csv'); print('copied')
else:
    !python scripts/build_manifest.py

## Cell 6 — install deps + GPU check (GPU must NOT be empty)

In [ ]:
!pip -q install -r requirements.txt
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 7 — DATA PATH CHECK ⛔ (stop if not 0)

In [ ]:
import pandas as pd, os
df=pd.read_csv('data/manifest.csv')
missing=[p for p in df['image_path'].head(300) if not os.path.exists(p)]
print('missing:', len(missing), '/ 300')
print('>>> OK: continue.' if not missing else f'>>> STOP. example: {missing[0]} ; folders: {os.listdir("data/raw")}')

## Cell 8 — pipeline on 3 models (CRASH-SAFE, LIVE, resumable)
If it disconnects: reconnect, re-run Cells 1-7 then this cell. Finished models skipped; a half-trained model resumes from its last epoch.

In [ ]:
# epochs=12 -> split as 6 frozen + 6 fine-tune per transfer model (lenet5 gets full 12).
# This is a FAST go/no-go: models only need to beat chance + differ in shortcut reliance,
# not fully converge, to tell us whether SRC predicts collapse. Bump to 20+ for the full run.
MODELS='lenet5 densenet201 resnet50'
BACKUP='/content/drive/MyDrive/cxr_data/backup'
!python -u scripts/run_pipeline.py --models $MODELS --epochs 12 --batch-size 64 --backup-dir $BACKUP

## Cell 9 — READ THE §8b VERDICT (decide threshold BEFORE looking)

In [ ]:
import json
c=json.load(open('results/c3_coupling.json'))
print(json.dumps(c,indent=2))
if c.get('n_models',0)>=3 and 'delta_acc' in c:
    r2=c['delta_acc']['r2']; r=c['delta_acc']['r']
    print(f'\nSRC -> delta_acc:  r={r:+.3f}  R2={r2:.3f}')
    print('GO -> colab_full.ipynb (spine=METHOD)' if (r>0 and r2>=0.3) else 'WEAK/NO-GO -> spine=AUDIT (§8b)')
else:
    print('Need 3 models with valid SRC + cross-source results.')

## Cell 10 — results are ALREADY on Drive (nothing to copy)

In [ ]:
import os
print('on Drive:', os.listdir('results'))